# MahjongMaster - treino com interface

Este notebook nao usa Google Drive. Ele foi feito para rodar no workspace local do VSCode/Jupyter, ou no Google Colab conectado a um **runtime local**.

Importante: um runtime remoto do Colab nao consegue ler automaticamente `C:/Codes/MahjongMaster/dataset` do seu PC. Para usar o dataset direto do seu computador, conecte o Colab ao runtime local ou abra este notebook pelo VSCode/Jupyter dentro do projeto.


In [ ]:
import sys
from pathlib import Path

print('Python:', sys.version)
try:
    import torch
    print('CUDA disponivel:', torch.cuda.is_available())
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            print(f'GPU {index}:', torch.cuda.get_device_name(index))
except Exception as error:
    print('Torch ainda nao carregado:', error)


In [ ]:
import os
import subprocess
import threading
import time
from pathlib import Path

try:
    import ipywidgets as widgets
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets'])
    import ipywidgets as widgets

from IPython.display import display, clear_output

IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}
TRAIN_PROCESS = None
TRAIN_THREAD = None


def find_project_root() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)
    for raw in ('C:/Codes/MahjongMaster', '/content/MahjongMaster'):
        candidates.append(Path(raw))
    for candidate in candidates:
        if (candidate / 'scripts' / 'train_yolo.py').exists() and (candidate / 'data' / 'mahjong_soul.yaml').exists():
            return candidate
    return cwd


def count_files(directory: Path, extensions: set[str]) -> int:
    if not directory.exists():
        return 0
    return sum(1 for path in directory.iterdir() if path.suffix.lower() in extensions)


def dataset_summary(root: Path) -> str:
    dataset = root / 'dataset'
    parts = []
    for split in ('train', 'val', 'test'):
        images = count_files(dataset / 'images' / split, IMAGE_EXTENSIONS)
        labels = count_files(dataset / 'labels' / split, {'.txt'})
        parts.append(f'{split}: {images} imgs/{labels} labels')
    return ' | '.join(parts)


def available_devices() -> list[tuple[str, str]]:
    items = [('CPU', 'cpu')]
    try:
        import torch
        if torch.cuda.is_available():
            for index in range(torch.cuda.device_count()):
                items.append((f'GPU {index}: {torch.cuda.get_device_name(index)}', str(index)))
    except Exception:
        pass
    return items


def available_models(root: Path) -> list[tuple[str, str]]:
    defaults = [
        ('yolo11n.pt', 'Nano - mais leve/rapido'),
        ('yolo11s.pt', 'Small - equilibrio inicial'),
        ('yolo11m.pt', 'Medium - mais preciso'),
        ('yolo11l.pt', 'Large - pesado/preciso'),
        ('yolo11x.pt', 'XLarge - mais pesado'),
        ('rtdetr-l.pt', 'RT-DETR Large'),
        ('rtdetr-x.pt', 'RT-DETR XLarge'),
    ]
    seen = {name for name, _description in defaults}
    items = [(f'{name} ({description})', name) for name, description in defaults]
    for path in sorted(root.glob('*.pt')):
        if path.name not in seen:
            items.append((f'{path.name} (arquivo local)', path.name))
    return items


def safe_run_name(text: str) -> str:
    safe = ''.join(char if char.isalnum() or char in ('-', '_') else '_' for char in text).strip('_')
    return safe or 'treino_colab_local'


project_root = find_project_root()
root_input = widgets.Text(value=str(project_root), description='Workspace', layout=widgets.Layout(width='720px'))
model_dropdown = widgets.Dropdown(options=available_models(project_root), value='yolo11n.pt', description='Modelo')
epochs_input = widgets.IntText(value=800, description='Epocas')
imgsz_input = widgets.IntText(value=1600, description='Imagem')
batch_input = widgets.IntText(value=5, description='Batch')
patience_input = widgets.IntText(value=0, description='Patience')
device_dropdown = widgets.Dropdown(options=available_devices(), description='Device')
run_name_input = widgets.Text(value='', description='Nome', placeholder='Automatico se vazio', layout=widgets.Layout(width='460px'))
install_button = widgets.Button(description='Instalar deps', button_style='')
refresh_button = widgets.Button(description='Verificar dataset', button_style='info')
start_button = widgets.Button(description='Iniciar treino', button_style='success')
stop_button = widgets.Button(description='Parar', button_style='danger')
tensorboard_button = widgets.Button(description='TensorBoard', button_style='')
status_html = widgets.HTML()
output = widgets.Output(layout=widgets.Layout(border='1px solid #334155', max_height='520px', overflow_y='auto'))


def current_root() -> Path:
    return Path(root_input.value).expanduser().resolve()


def default_run_name() -> str:
    return f'{epochs_input.value}e_{Path(model_dropdown.value).stem}_{imgsz_input.value}p_{batch_input.value}b'


def selected_run_name() -> str:
    return safe_run_name(run_name_input.value.strip() or default_run_name())


def build_train_command() -> list[str]:
    root = current_root()
    return [
        sys.executable,
        '-u',
        str(root / 'scripts' / 'train_yolo.py'),
        '--model', str(model_dropdown.value),
        '--data', str(root / 'data' / 'mahjong_soul.yaml'),
        '--epochs', str(int(epochs_input.value)),
        '--imgsz', str(int(imgsz_input.value)),
        '--batch', str(int(batch_input.value)),
        '--patience', str(int(patience_input.value)),
        '--device', str(device_dropdown.value),
        '--project', str(root / 'runs' / 'detect'),
        '--name', selected_run_name(),
        '--exist-ok',
    ]


def refresh_status(*_args) -> None:
    root = current_root()
    ok_script = (root / 'scripts' / 'train_yolo.py').exists()
    ok_data = (root / 'data' / 'mahjong_soul.yaml').exists()
    ok_dataset = (root / 'dataset' / 'images' / 'train').exists()
    model_dropdown.options = available_models(root)
    if model_dropdown.value not in [value for _label, value in model_dropdown.options]:
        model_dropdown.value = 'yolo11n.pt'
    command = ' '.join(f'"{part}"' if ' ' in part else part for part in build_train_command())
    status_html.value = (
        f'<b>Workspace:</b> {root}<br>'
        f'<b>Status:</b> script {"OK" if ok_script else "NAO"} | data yaml {"OK" if ok_data else "NAO"} | dataset {"OK" if ok_dataset else "NAO"}<br>'
        f'<b>Dataset:</b> {dataset_summary(root)}<br>'
        f'<b>Run:</b> {selected_run_name()}<br>'
        f'<b>Comando:</b> <code>{command}</code>'
    )


def install_deps(_button) -> None:
    root = current_root()
    with output:
        print('[DEPS] Instalando requirements-train.txt...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(root / 'requirements-train.txt'), 'ipywidgets'])
    with output:
        print('[DEPS] Pronto.')


def stream_training(command: list[str], root: Path) -> None:
    global TRAIN_PROCESS
    start_button.disabled = True
    stop_button.disabled = False
    try:
        with output:
            print('[TRAIN] Workspace:', root)
            print('[TRAIN] Iniciando:', ' '.join(command))
        TRAIN_PROCESS = subprocess.Popen(
            command,
            cwd=str(root),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert TRAIN_PROCESS.stdout is not None
        for line in TRAIN_PROCESS.stdout:
            with output:
                print(line, end='')
        return_code = TRAIN_PROCESS.wait()
        with output:
            print(f'\n[TRAIN] Finalizado com codigo {return_code}.')
            print('[TRAIN] best.pt esperado em:', root / 'runs' / 'detect' / selected_run_name() / 'weights' / 'best.pt')
            print('[TRAIN] teste final esperado em:', root / 'runs' / 'detect' / selected_run_name() / 'test')
    finally:
        TRAIN_PROCESS = None
        start_button.disabled = False
        stop_button.disabled = True
        refresh_status()


def start_training(_button) -> None:
    global TRAIN_THREAD
    if TRAIN_PROCESS is not None:
        with output:
            print('[TRAIN] Ja existe um treino rodando.')
        return
    root = current_root()
    if not (root / 'scripts' / 'train_yolo.py').exists():
        with output:
            print('[ERRO] Workspace invalido: scripts/train_yolo.py nao encontrado.')
        return
    if not (root / 'dataset' / 'images' / 'train').exists():
        with output:
            print('[ERRO] Dataset local nao encontrado em dataset/images/train.')
        return
    command = build_train_command()
    TRAIN_THREAD = threading.Thread(target=stream_training, args=(command, root), daemon=True)
    TRAIN_THREAD.start()


def stop_training(_button) -> None:
    if TRAIN_PROCESS is None:
        return
    with output:
        print('[TRAIN] Parando processo...')
    TRAIN_PROCESS.terminate()


def start_tensorboard(_button) -> None:
    root = current_root()
    logdir = root / 'runs' / 'detect'
    with output:
        print('[TB] Logdir:', logdir)
        print('[TB] Se estiver no VSCode/Jupyter local, abra http://localhost:6006')
    subprocess.Popen([sys.executable, '-m', 'tensorboard.main', '--logdir', str(logdir), '--host', '127.0.0.1', '--port', '6006'])


install_button.on_click(install_deps)
refresh_button.on_click(lambda button: refresh_status())
start_button.on_click(start_training)
stop_button.on_click(stop_training)
tensorboard_button.on_click(start_tensorboard)
for widget in (root_input, model_dropdown, epochs_input, imgsz_input, batch_input, patience_input, device_dropdown, run_name_input):
    widget.observe(refresh_status, names='value')

stop_button.disabled = True
refresh_status()

controls = widgets.VBox([
    root_input,
    widgets.HBox([model_dropdown, device_dropdown]),
    widgets.HBox([epochs_input, imgsz_input, batch_input, patience_input]),
    run_name_input,
    widgets.HBox([refresh_button, install_button, start_button, stop_button, tensorboard_button]),
    status_html,
    output,
])
display(controls)
